# Datastädning - utforskning med spaCy

- titta efter personuppgifter (namn, telefonnummer och emailadress) och ta bort (med spaCy)
- titta efter jobbannonser med samma ID och ta bort duplicerade

Notera att JobTechs release notes nämner att de redan har en extra kontroll som tar bort personuppgifter från historiska annonser, så det vi letar efter är förmodligen endast rester och inte allt. Det gör att en enkel metod som fångar det mesta bör räcka.

## 1. Ladda och titta på hur annonser slutar

Personuppgifter återfinns ofta i kontaktuppgifter i slutet av annonstexterna. 

In [4]:
import pandas as pd

ads = pd.read_json("../data/raw_ads_2511.json")
print(len(ads))

# slumpa 10 annonser och titta på de sista 300 tecknen
sample = ads.dropna(subset=["description"]).sample(10, random_state=1)
for _, row in sample.iterrows():
    print(row["id"], "...", row["description"][-300:])
    print("-" * 60)

11377
30953723 ... okumenterad arbetslivserfarenhet från myndighet eller statligt affärsverk. Certifiering inom kravhantering, exempelvis IREB, REQB eller motsvarande. Vi erbjuderFast anställning hos Avaron AB
Tjänstepension
Friskvårdsbidrag på 5 000 kr per år
Ansökan
Vi tillsätter löpande – sök gärna så snart du kan.
------------------------------------------------------------
29454318 ... engage and motivate stakeholders at all levels. Problem-solving mindset with the ability to navigate challenges and find effective solutions. Proactive, self-driven, and able to work independently while taking an educational and coaching approach. Experience in the pharmaceutical industry is a plus.
------------------------------------------------------------
30007130 ... , grow, and shape the future together. Spana in vårt Insta-flöde för att få en bild av hur det är att jobba hos oss genom våra kollegor. Klicka på bilden för att komma till Instagram-kontot. Helt ofiltrerat har vi samlat in omdömen

Utskrifter visar att inga personuppgifter dyker upp i någon av exemplerna. Kan bero på att det antingen inte finns några personuppgifter alls i exempelannonserna (redan borttagna av JobTech) eller att de finns någon annanstans än i slutet av annonstexten. 

## 2. Personuppgifter som spaCy hittar

Kör modellen på ett litet urval och titta på alla entitetstyper, inte bara personer för att identifiera personnamn.

Vad vi letar efter: 

- Riktiga namn: Fångar spaCy de personer du själv ser i slutet av annonserna i steg 1?
- Falska träffar: Vilka ord är det?
- Missade namn: Står det ett namn i slutet av någon annons i steg 1 som inte finns bland träffarna?
- Positionerna. Hur fördelar sig POS för de riktiga namnen?

In [7]:
import spacy
from collections import Counter

nlp = spacy.load("sv_core_news_sm", disable=["parser", "lemmatizer"])
print(nlp.get_pipe("ner").labels)   # vilka etiketter finns?

sample50 = ads.sample(50, random_state=1)

counts = Counter()
per_hits = []   # (annons-id, träff, var i texten 0-1, omgivande text)

for _, row in sample50.iterrows():
    text = row["description"]
    doc = nlp(text)
    for ent in doc.ents:
        counts[ent.label_] += 1
        if ent.label_ == "PRS":     
            pos = ent.start_char / len(text)
            context = text[max(ent.start_char - 40, 0): ent.end_char + 40].replace("\n", " ")
            per_hits.append((row["id"], ent.text, round(pos, 2), context))

print(counts)
print(len(per_hits), "personträffar")
print(Counter(h[1] for h in per_hits).most_common(30))
for h in per_hits[:30]:
    print(h)

('EVN', 'LOC', 'MSR', 'OBJ', 'ORG', 'PRS', 'TME', 'WRK')
Counter({'PRS': 231, 'ORG': 176, 'LOC': 114, 'WRK': 13, 'TME': 5, 'MSR': 4, 'OBJ': 2, 'EVN': 1})
231 personträffar
[('solutions', 18), ('with', 14), ('Avaron AB', 9), ('With', 9), ('workshops', 5), ('careers', 4), ('Europe', 4), ('Collaborate with', 4), ('Experience with', 4), ('Intervjuer', 3), ('Visual Studio', 3), ('Collaborate', 3), ('Scaled Agile Framework', 3), ('Eventful', 3), ('LinkedIn', 2), ('Server Subscription Edition', 2), ('Familiarity with', 2), ('Azure Integration Services', 2), ('Logic Apps', 2), ('Function Apps', 2), ('Azure', 2), ('Data Scientist', 2), ('Python', 2), ('Sales', 2), ('Kubernetes', 2), ('This', 2), ('About Rasulson Consulting', 2), ('Leda', 2), ('OpenID', 2), ('Ansök', 2)]
(30953723, 'workshops', 0.34, 'ng i projekt med IT-påverkan. Du håller workshops och samlar in funktionella och icke-fun')
(30953723, 'Avaron AB', 0.97, 'arande. Vi erbjuderFast anställning hos Avaron AB Tjänstepension Friskvård

### Vad resultatet visar

Av de träffar som syns är ingen ett personnamn. De vanligaste är solutions, with, workshops, careers, Europe, företagsnamn som Avaron AB, och teknikord som Azure, Python, Kubernetes och Visual Studio.

Noll riktiga namn bland träffarna kan betyda två olika saker:

1. Namnen finns inte i texten, eftersom JobTech tagit bort dem.
2. Namnen finns, men modellen är för dålig på jobbannonser för att hitta dem.

På det stora hela är spaCy inte speciellt användbar för att identifiera personnamn i annonserna. Valde därför att inte utforska email och telefonnummer vidare då spaCy ej är lämpligt att använda i detta fall. 

## 3. ID-dubletter

### Kolla efter dubletter

In [3]:
import pandas as pd

ads = pd.read_json("../data/raw_ads_2511.json", dtype={"id": str, "version_id": str})
print(len(ads), "rader totalt")
print(ads["id"].nunique(), "unika annons id")
print(ads["id"].duplicated().sum(), "extra rader med upprepat id")
print(ads["version_id"].duplicated().sum(), "upprepade version_id (borde vara 0)")

11377 rader totalt
6800 unika annons id
4577 extra rader med upprepat id
4320 upprepade version_id (borde vara 0)


Ger sista raden något annat än 0 har samma post hämtats två gånger. Det kan hända om en annons ligger precis på gränsen mellan två månadsfönster. Bugg i hämtningen som behöver lösas eller ok och ta bort här istället?

### Titta på några dublett-exempel

In [4]:
dup_ids = ads["id"].duplicated(keep=False)   # markerar ALLA rader i en dubblettgrupp
dups = ads[dup_ids].sort_values(["id", "published"])
print(dups[["id", "version_id", "published", "title", "employer"]].head(10))

# Skiljer sig texten mellan versionerna?
text_variants = dups.groupby("id")["description"].nunique()
print((text_variants > 1).sum(), "av", dups["id"].nunique(),
      "id har olika text i olika versioner")

           id                                version_id            published  \
477  29354585  26008c8a15f486a5b9163029fddd5d7fc3c03cb4  2025-01-02T10:15:14   
478  29354585  f0883b87209ad585259381a396b05e5618ea3d93  2025-01-02T10:15:14   
451  29361752  5fddf1db951f3374f7792910c4f425650ce236e6  2025-01-07T13:27:42   
452  29361752  27b784e84eb1a5703befb304f9f76c3a3c8797db  2025-01-07T13:27:42   
444  29362368  98e4dc9c1ec0419c0fcef0319b64909342df680d  2025-01-07T15:11:03   
445  29362368  aa58e8f5ebbe56dd9dbc558eea70eb14085eb6de  2025-01-07T15:11:03   
442  29362443  4acb1a3da382edaf553e92b57524ad381d704d85  2025-01-07T15:28:34   
443  29362443  a335ce8a599380486cb101644acedcc27bef5d5d  2025-01-07T15:28:34   
430  29363892  de9e985fb72fb75014b4c4f676ce7c8bee93db9e  2025-01-08T09:56:04   
431  29363892  fdde3617b56e6d11100f6ad997d8606bd3dfbbc5  2025-01-08T09:56:04   

                                            title              employer  
477                                 Projektle

### Ta bort dubeletterna

To do: lägg till kod